# <span style="color:red"> Lecture 21: APIs </span>

<font size = "4">

In the previous class we:

- Converted between JSON text and Python dictionaries and lists
- Navigated the nested structure of chess.com data
- Wrote a function, `stats_to_rows`, that turns a player's stats into rows of a DataFrame

In this class we will:

- Request data from a web API with the `requests` library
- Check whether a request worked using its **status code**
- Be a good API citizen: identify ourselves, pause between requests, and **cache** results
- Build a class that talks to the chess.com API
- Use it to **update** a DataFrame with fresh data

In [1]:
import json
import time

import requests
import pandas as pd
import matplotlib.pyplot as plt

<font size = "4">

**Code from Lecture 20**

In [2]:
time_controls2 = ["chess_bullet", "chess_blitz", "chess_rapid"]
# We're skipping "chess_daily" since some of our data will be incomplete for it

def stats_to_rows(username, stats):
    """One row (dictionary) per time control in a player's stats."""
    rows = []
    for time_control in time_controls2:
        if time_control in stats:
            info = stats[time_control]
            rows.append({"username":    username,
                         "time_class":  time_control.replace("chess_", ""),
                         "rating":      info["last"]["rating"],
                         "rd":          info["last"]["rd"],
                         "rating_date": info["last"]["date"],
                         "best_rating": info["best"]["rating"],
                         "wins":        info["record"]["win"],
                         "losses":      info["record"]["loss"],
                         "draws":       info["record"]["draw"]})
    return rows

# <span style="color:red"> I. What is an API? </span>

<font size = "4">

- An API (Application Programming Interface) is a set of rules for how one program can ask another program for data.
- With a web API, you request a URL and get data back, usually as JSON.
- The JSON files we used last class were downloaded from the chess.com API.

Every chess.com API URL has two parts:

```
https://api.chess.com/pub/player/hikaru/stats
\______________________/\________________/
        base URL                 path
```

- The base URL is the same for every request.
- The path says which data we want. The API's documentation lists the paths it offers:

| Path | Data |
| --- | --- |
| `/player/{username}` | profile |
| `/player/{username}/stats` | ratings for every time control |
| `/player/{username}/games/{year}/{month}` | every game the player finished that month |
| `/titled/{title}` | every account with a title, e.g. GM |

Some APIs also take query parameters after a `?`, written as `name=value` pairs separated by `&`. For example, the Formula One API in the exercises:

```
https://api.jolpi.ca/ergast/f1/2025/results.json?limit=100&offset=0
```

**Try it yourself!** Paste https://api.chess.com/pub/player/hikaru/stats into your web browser. The browser shows the same JSON we loaded from a file last class.

### <span style="color:red"> Documentation </span>

<font size = "4">

- chess.com API documentation: https://www.chess.com/news/view/published-data-api
- Jolpica F1 API documentation: https://github.com/jolpica/jolpica-f1
- The requests library: https://requests.readthedocs.io

### <span style="color:red"> Contingency Plan for API errors </span>


In [ ]:
# # OFFLINE MODE: only run this cell if the chess.com API isn't working for you when you are trying to follow along with the notes

# # Quickly uncomment with ctrl (or cmd on Mac) + A and then ctrl+/

# with open("chess_raw_data/chess_cache.json", "r", encoding = "utf-8") as file:
#     saved_answers = json.load(file)
# with open("f1_api_data/f1_cache.json", "r", encoding = "utf-8") as file:
#     saved_answers.update(json.load(file))

# class SavedResponse:
#     def __init__(self, url):
#         self.url = url
#         if url in saved_answers:
#             self.status_code = 200
#         else:
#             self.status_code = 404

#     def __repr__(self):
#         return f"<Response [{self.status_code}]>"

#     def json(self):
#         return saved_answers[self.url]

# def offline_get(url, headers = None):
#     return SavedResponse(url)

# requests.get = offline_get

# <span style="color:red"> II. Requesting Data with requests </span>

<font size = "4">

- `requests.get(url)` does what your browser just did, from Python.
- Every request also sends a **"User-Agent"**: a short label saying what program is asking. chess.com asks programs to use a label that describes them.

In [3]:
HEADERS = {"User-Agent": "DATASCI185 professor notebook (Emory University)"}

url = "https://api.chess.com/pub/player/hikaru/stats"
response = requests.get(url, headers = HEADERS)

# First question to always ask with a new function:
# what kind of thing does it output?
print(type(response))
print(response)

<class 'requests.models.Response'>
<Response [200]>


<font size = "4">

The number in `<Response [200]>` is the **status code**. It tells us whether the request worked:

| Status code | Meaning |
| --- | --- |
| 200 | Success |
| 404 | Not found: the URL is wrong, or there is no data |
| 429 | Too many requests: slow down! |
| 500-599 | A problem on the server's side |

In [5]:
print(response.status_code)

200


<font size = "4">

The `.json()` method converts the response into Python dictionaries and lists, just like `json.load` did for our files last class:

In [4]:
stats = response.json()

print(type(stats))
print(stats["chess_blitz"]["last"])

<class 'dict'>
{'rating': 3454, 'date': 1794938417, 'rd': 30}


<font size = "4">

This is the same kind of dictionary we loaded from a file last class, so our function from Lecture 20 works unchanged:

In [7]:
display(pd.DataFrame(stats_to_rows("Hikaru", stats)))

,username,time_class,rating,rd,rating_date,best_rating,wins,losses,draws
0,Hikaru,bullet,3402,28,1794860103,3570,17334,2562,1197
1,Hikaru,blitz,3454,30,1794938417,3503,36242,5571,4396
2,Hikaru,rapid,2838,44,1786796329,2927,205,69,220


<font size = "4">

Not every request works. Always check the status code before using the data:

In [6]:
for username in ["Hikaru", "MagnusCarlsen", "not_a_real_user_185"]:
    url = f"https://api.chess.com/pub/player/{username.lower()}/stats"
    response = requests.get(url, headers = HEADERS)
    print(username, response.status_code)

Hikaru 200
MagnusCarlsen 404
not_a_real_user_185 404


<font size = "4">

- The made-up username returns 404: there is no such player.
- When these notes were written, Magnus Carlsen's stats also returned 404, even though his profile (`/player/magnuscarlsen`) works. Real APIs don't always have data for everything!

<font size = "4">

Save to a JSON file for easier readability

- Save the response with 4-space indentation, then open it in VS Code.

In [8]:
with open("my_hikaru_stats.json", "w", encoding = "utf-8") as file:
    json.dump(stats, file, indent = 4)

# <span style="color:red"> III. Being a Good API Citizen </span>

<font size = "4">

- Every request uses the server's time and resources. Most APIs limit how many requests you can make, and answer with status code 429 if you make too many.
- chess.com asks that you make requests one at a time. The Formula One API allows only a few requests per second and a few hundred per hour.
- A whole classroom sending requests from the same network can hit these limits quickly.

Two good habits:

1. **Pause** between requests
2. **Cache** results: save each answer, so you never request the same data twice

<font size = "4">

**Pausing:** `time.sleep(seconds)` makes Python wait before running the next line.

In [9]:
for username in ["Hikaru", "GMWSO"]:
    response = requests.get(f"https://api.chess.com/pub/player/{username.lower()}",
                            headers = HEADERS)
    print(username, response.status_code)
    time.sleep(0.5)    # wait half a second before the next request

Hikaru 200
GMWSO 200


<font size = "4">

**Caching:** a dictionary whose keys are URLs and whose values are the JSON we got back.

- Before making a request, check whether the URL is already in the dictionary.
- If it is, reuse the stored answer: no internet needed.
- If not, make the request, store the answer, and return it.

In [10]:
cache = {}

def get_json(url):
    """Return the JSON at url, using the cache when possible. None if the request fails."""
    if url in cache:
        return cache[url]

    response = requests.get(url, headers = HEADERS)
    time.sleep(0.5)

    if response.status_code != 200:
        print("  status", response.status_code, "for", url)
        return None

    cache[url] = response.json()
    return cache[url]

In [11]:
url = "https://api.chess.com/pub/player/gmwso/stats"

# time.time() is the current time, in seconds
start = time.time()
stats_so = get_json(url)
print("First request: ", round(time.time() - start, 3), "seconds")

start = time.time()
stats_so = get_json(url)
print("Second request:", round(time.time() - start, 3), "seconds")

print("URLs in cache:", list(cache.keys()))

First request:  0.598 seconds
Second request: 0.0 seconds
URLs in cache: ['https://api.chess.com/pub/player/gmwso/stats']


<font size = "4">

- The cache is a dictionary of JSON, so it can be saved as a .json file and loaded again next time.
- **The downside:** the cache doesn't know when the data has changed. If Wesley So plays a game, `get_json` still returns his old rating. We'll need a way to skip the cache when we want fresh data.

# <span style="color:red"> IV. A Class That Talks to the API </span>

<font size = "4">

The headers, the pause, the cache, and the request count all belong together, which makes them a good fit for a custom class:

- `BASE_URL` is a **class variable**: the same for every client.
- `get_json` takes a **path** instead of a full URL. It has a parameter `use_cache` with a default of `True`; set it to `False` to force a fresh request.
- `get_profile` and `get_stats` are short methods built on top of `get_json`, so we don't have to remember the paths.

The program requesting data is called the **client**.

The program responding with data is called the **server**.

In [13]:
class ChessComClient:
    """Requests data from the chess.com API, with a pause and a cache."""

    BASE_URL = "https://api.chess.com/pub"

    def __init__(self, user_agent, sleep_seconds = 0.5):
        self.headers = {"User-Agent": user_agent}
        self.sleep_seconds = sleep_seconds
        self.cache = {}
        self.num_requests = 0

    def __str__(self):
        return f"ChessComClient: {self.num_requests} requests made, {len(self.cache)} answers cached"

    # ---------- getters ----------

    def get_num_requests(self):
        return self.num_requests

    # ---------- requesting ----------

    def get_json(self, path, use_cache = True):
        """JSON for BASE_URL + path, e.g. path = "/player/hikaru". None if the request fails."""
        url = ChessComClient.BASE_URL + path
        if use_cache and url in self.cache:
            return self.cache[url]

        response = requests.get(url, headers = self.headers)
        self.num_requests = self.num_requests + 1
        time.sleep(self.sleep_seconds)

        if response.status_code != 200:
            print(f"  status {response.status_code} for {url}")
            return None

        self.cache[url] = response.json()
        return self.cache[url]

    def get_profile(self, username, use_cache = True):
        return self.get_json(f"/player/{username.lower()}", use_cache)

    def get_stats(self, username, use_cache = True):
        return self.get_json(f"/player/{username.lower()}/stats", use_cache)

    # ---------- saving and loading the cache ----------

    def save_cache(self, path):
        with open(path, "w", encoding = "utf-8") as file:
            json.dump(self.cache, file)

    def load_cache(self, path):
        with open(path, "r", encoding = "utf-8") as file:
            self.cache = json.load(file)

In [14]:
client = ChessComClient("DATASCI185-course-notebook (Emory University)")
print(client)

profile_so = client.get_profile("GMWSO")
print(profile_so["name"], profile_so["title"])
print(client)

profile_so = client.get_profile("GMWSO")     # from the cache: no new request
print(client)

profile_so = client.get_profile("GMWSO", use_cache = False)     # forces a new request
print(client)

ChessComClient: 0 requests made, 0 answers cached
Wesley So GM
ChessComClient: 1 requests made, 1 answers cached
ChessComClient: 1 requests made, 1 answers cached
ChessComClient: 2 requests made, 1 answers cached


# <span style="color:red"> V. Updating a DataFrame </span>

<font size = "4">

"chess_clean_data/ratings_start.csv" contains our eight players' ratings from a few days ago, built with `stats_to_rows`. Since then they've played more games, so their ratings have changed. Our goal: **update the DataFrame with fresh data from the API.**

Three pieces work together:

- `ChessComClient` requests the data
- `stats_to_rows` turns it into rows
- A new class, `RatingsTracker`, holds the DataFrame and uses the other two to keep it up to date. Just like a `LabSection` held its `Student` objects in Lecture 16, a `RatingsTracker` holds a `ChessComClient`.

In [15]:
df_start = pd.read_csv("chess_clean_data/ratings_start.csv")
display(df_start)

,username,time_class,rating,rd,rating_date,best_rating,wins,losses,draws
0,Hikaru,bullet,3424,26,1794077982,3570.0,17282,2552,1192
1,Hikaru,blitz,3464,30,1794334422,3503.0,36193,5562,4391
2,Hikaru,rapid,2838,44,1786796329,2927.0,205,69,220
3,Hikaru,daily,2239,103,1770563021,2464.0,73,11,4
4,GHANDEEVAM2003,bullet,3238,33,1793473171,3360.0,9848,6307,1376
5,GHANDEEVAM2003,blitz,3279,35,1793935271,3344.0,5351,3130,1129
6,GHANDEEVAM2003,rapid,2791,46,1786710907,2829.0,182,84,94
7,GHANDEEVAM2003,daily,1187,240,1679993876,1309.0,1,1,0
8,Firouzja2003,bullet,3386,34,1793473197,3521.0,12276,7835,1454
9,Firouzja2003,blitz,3391,35,1794159322,3402.0,8377,4747,1885


<font size = "4">

One new Pandas tool: `pd.concat` stacks DataFrames on top of each other ("concatenation", mentioned in Lecture 18a). `ignore_index = True` renumbers the rows 0, 1, 2, ...

In [16]:
df_a = pd.DataFrame({"username": ["Hikaru", "GMWSO"], "rating": [3464, 3267]})
df_b = pd.DataFrame({"username": ["DenLaz"], "rating": [3333]})

display(pd.concat([df_a, df_b], ignore_index = True))

,username,rating
0,Hikaru,3464
1,GMWSO,3267
2,DenLaz,3333


<font size = "4">

`RatingsTracker` has two methods that update its DataFrame:

- `add_player` **adds rows**: it requests one player's stats and stacks the new rows below the existing ones.
- `refresh` **updates the values**: it requests fresh stats for every player (skipping the cache), then merges the old and new ratings on **two** columns, "username" and "time_class", to see what changed. `suffixes` names the two "rating" columns "rating_before" and "rating_after".

In [17]:
class RatingsTracker:
    """Keeps a DataFrame of ratings up to date using a ChessComClient."""

    def __init__(self, client, csv_path):
        self.client = client
        self.ratings = pd.read_csv(csv_path)
        # stats_to_rows skips daily ratings, so drop them from the starting data too
        self.ratings = self.ratings.query("time_class != 'daily'").reset_index(drop = True)

    def __str__(self):
        return f"RatingsTracker: {len(self.get_usernames())} players, {len(self.ratings)} rows"

    # ---------- getters ----------

    def get_ratings(self):
        return self.ratings

    def get_usernames(self):
        return list(self.ratings["username"].unique())

    # ---------- updating ----------

    def add_player(self, username):
        """Request one player's stats and add their rows to the DataFrame."""
        if username in self.get_usernames():
            print(username, "is already being tracked")
            return

        stats = self.client.get_stats(username)
        if stats is None:
            print("Could not add", username)
            return

        new_rows = pd.DataFrame(stats_to_rows(username, stats))
        self.ratings = pd.concat([self.ratings, new_rows], ignore_index = True)

    def refresh(self):
        """Request fresh stats for every player. Returns a DataFrame comparing old and new ratings."""
        rows = []
        for username in self.get_usernames():
            stats = self.client.get_stats(username, use_cache = False)
            if stats is not None:
                rows = rows + stats_to_rows(username, stats)

        if len(rows) == 0:
            print("No new data: is the internet working?")
            return None
        df_new = pd.DataFrame(rows)

        comparison = pd.merge(self.ratings[["username", "time_class", "rating"]],
                              df_new[["username", "time_class", "rating"]],
                              on = ["username", "time_class"],
                              how = "left",
                              suffixes = ("_before", "_after"))
        comparison["change"] = comparison["rating_after"] - comparison["rating_before"]

        self.ratings = df_new
        return comparison

In [18]:
tracker = RatingsTracker(client, "chess_clean_data/ratings_start.csv")
print(tracker)

RatingsTracker: 8 players, 24 rows


<font size = "4">

Adding players:

In [19]:
tracker.add_player("Hikaru")           # already there
tracker.add_player("MagnusCarlsen")    # stats not available
tracker.add_player("nihalsarin")       # Nihal Sarin

print(tracker)
display(tracker.get_ratings().tail())

Hikaru is already being tracked
  status 404 for https://api.chess.com/pub/player/magnuscarlsen/stats
Could not add MagnusCarlsen
RatingsTracker: 9 players, 27 rows


,username,time_class,rating,rd,rating_date,best_rating,wins,losses,draws
22,LyonBeast,blitz,3247,34,1794256726,3315.0,2366,830,572
23,LyonBeast,rapid,2811,47,1780850354,2886.0,109,46,91
24,nihalsarin,bullet,3282,36,1791653112,3458.0,15161,7498,1920
25,nihalsarin,blitz,3331,27,1793871514,3393.0,17490,9306,3205
26,nihalsarin,rapid,2767,46,1786716408,2806.0,207,93,87


<font size = "4">

Refreshing every player's ratings:

In [20]:
changes = tracker.refresh()
print(client)
display(changes)

  status 404 for https://api.chess.com/pub/player/firouzja2003/stats
ChessComClient: 13 requests made, 9 answers cached


,username,time_class,rating_before,rating_after,change
0,Hikaru,bullet,3424,3402.0,-22.0
1,Hikaru,blitz,3464,3451.0,-13.0
2,Hikaru,rapid,2838,2838.0,0.0
3,GHANDEEVAM2003,bullet,3238,3238.0,0.0
4,GHANDEEVAM2003,blitz,3279,3279.0,0.0
5,GHANDEEVAM2003,rapid,2791,2791.0,0.0
6,Firouzja2003,bullet,3386,NaN,NaN
7,Firouzja2003,blitz,3391,NaN,NaN
8,Firouzja2003,rapid,2802,NaN,NaN
9,FabianoCaruana,bullet,3133,3133.0,0.0


<font size = "4">

Save the updated DataFrame and the cache for next time:

In [21]:
tracker.get_ratings().to_csv("chess_clean_data/ratings_updated.csv", index = False)
client.save_cache("my_chess_cache.json")

<font size = "4">

**Why a class *and* a function?**

- `ChessComClient` only knows how to **get** data. It doesn't know anything about DataFrames.
- `stats_to_rows` only knows how to **convert** data. It doesn't care whether the stats came from a file (Lecture 20) or the internet (today).
- `RatingsTracker` **combines** them. Keeping these jobs separate means each piece can be tested on its own, and reused: the exercises use the same client for game archives.

# <span style="color:red"> Exercises </span>
<font size = "4">

**Checking your work:** after each exercise there is a cell of **tests**. Run it after your code. Each test uses the keyword `assert`:

```python
assert condition, "message"
```

- If condition is True, nothing happens.
- If condition is False, Python stops with an `AssertionError` and shows the message, which tells you what to fix.

If every test in a cell passes, the cell prints a message saying so.


<font size = "4">

**Code from the Lecture 20 exercises:** a function that turns one game into one row. (It has a few more keys than in Lecture 20, to match the .csv files used below.)

In [26]:
def game_to_row(game):
    """Flatten one chess.com game dictionary into one row."""
    accuracies = game.get("accuracies", {})

    row = {"url":            game["url"],
           "end_time":       game["end_time"],
           "time_class":     game["time_class"],
           "time_control":   game["time_control"],
           "rules":          game["rules"],
           "rated":          game["rated"],
           "white_username": game["white"]["username"],
           "white_rating":   game["white"]["rating"],
           "white_result":   game["white"]["result"],
           "white_accuracy": accuracies.get("white"),
           "black_username": game["black"]["username"],
           "black_rating":   game["black"]["rating"],
           "black_result":   game["black"]["result"],
           "black_accuracy": accuracies.get("black"),
           "eco":            game.get("eco", "")}
    return row

<font size = "4">

**Exercise 1 (adding a method):** `ChessComClient` has no method for game archives yet. The path for one month of a player's games is `/player/{username}/games/{year}/{month}`, where the month has two digits (e.g. "03" for March).

The cell below creates a class `MyChessComClient` that **inherits** from `ChessComClient` (Lecture 16), so it already has `BASE_URL`, `get_json`, `get_profile`, and `get_stats`. It adds one method, `get_games`, which is missing one line:

- Replace `""` with an f-string for the path, using `username.lower()` (like `get_stats` does), `year`, and `month`.
- Hint: in an f-string, `{month:02d}` writes an integer with two digits, so 3 becomes "03".

The rest of the cell creates `my_client`, requests Hikaru's games for October 2026, and prints the number of games.

In [ ]:
class MyChessComClient(ChessComClient):
    """ChessComClient with an extra method for game archives."""

    def get_games(self, username, year, month, use_cache = True):
        """JSON of every game a player finished in one month. None if the request fails."""
        path = f"/player/{username.lower()}/games/{year}/{month:02d}"    # <- solution
        return self.get_json(path, use_cache)


my_client = MyChessComClient("DATASCI185-course-notebook (Emory University)")

hikaru_october = my_client.get_games("Hikaru", 2026, 10)
print(len(hikaru_october["games"]))

583


In [23]:
# Tests for Exercise 1
assert type(hikaru_october) == dict, "get_games should return a dictionary. Did the request fail? Check the path."
assert "games" in hikaru_october, 'The result should have the key "games"'
assert "https://api.chess.com/pub/player/hikaru/games/2026/10" in my_client.cache, \
    "The path should look like /player/hikaru/games/2026/10"

# The username should be lowercase, and a one-digit month should get two digits
march = my_client.get_games("HIKARU", 2026, 3)
assert "https://api.chess.com/pub/player/hikaru/games/2026/03" in my_client.cache, \
    "Use username.lower(), and write the month with two digits: 3 should become 03"

# Every game in the October archive should have ended in October
for game in hikaru_october["games"]:
    assert pd.to_datetime(game["end_time"], unit = "s").month == 10, "Every game should have ended in October"

print("All tests for Exercise 1 passed!")

All tests for Exercise 1 passed!


<font size = "4">

The folder "chess_clean_data" contains a .csv of each player's games, made a few days ago, e.g. "games_hikaru.csv". Choose a player and store their username in the variable `username`, spelled as in Lecture 20 (e.g. "Hikaru", "GHANDEEVAM2003", "MagnusCarlsen").

In [39]:
username = "GHANDEEVAM2003"    # change this to the player you chose

<font size = "4">

**Exercise 2 (adding new games):**

- Read in `f"chess_clean_data/games_{username.lower()}.csv"` and call it `df_old`. Convert its "end_time" column to dates with `pd.to_datetime(..., unit = "s")`.
- Store the date of the most recent game, `df_old["end_time"].max()`, in a variable called `latest_game`, and print it.
- Use `my_client.get_games(...)` (from Exercise 1) to request the archive for the year and month of `latest_game` (`latest_game.year` and `latest_game.month`). Use the loop over `game_to_row` to create a DataFrame called `df_new` with one row per game, and convert its "end_time" column to dates.
- If today's month (`pd.Timestamp.today().month`) is different from the month of `latest_game`, also request today's month and add its games to `df_new`.
- The API sends "time_control" as text, but `pd.read_csv` read it as numbers. Convert the old column to text with `df_old["time_control"] = df_old["time_control"].astype(str)`.
- Stack `df_old` and `df_new` with `pd.concat` (use `ignore_index = True`), then remove duplicate games with `.drop_duplicates(subset = "url")`. Call the result `df_all`.
- Print the number of new games: `len(df_all) - len(df_old)`.

In [ ]:
# your code here

# not the solution, but it has ideas of a solution

# I'm not grapping neccissarily the latest game, I went to chess.com and made sure that Arjun had played games in Febuary 2023
# This data isn't in the cache, so get_jason (called by get_games has to reach out to the API even though use cache is set to True)

historical_games = my_client.get_games(username, 2023, 2)
print(type(historical_games))
print(historical_games.keys())
df_new = []
for g in historical_games["games"]:
    df_new.append(game_to_row(g))

df_new = pd.DataFrame(df_new)
display(df_new)

2026-10-06 15:07:57
<class 'dict'>
dict_keys(['games'])


,url,end_time,time_class,time_control,rules,rated,white_username,white_rating,white_result,white_accuracy,black_username,black_rating,black_result,black_accuracy,eco
0,https://www.chess.com/game/live/69182174931,1675444365,rapid,600+2,chess,True,Indianlad,2555,agreed,96.36,GHANDEEVAM2003,2706,agreed,96.84,https://www.chess.com/openings/Semi-Slav-Defen...
1,https://www.chess.com/game/live/69184021543,1675446018,rapid,600+2,chess,True,GHANDEEVAM2003,2717,win,88.50,Beca95,2522,checkmated,87.12,https://www.chess.com/openings/Philidor-Defens...
2,https://www.chess.com/game/live/69185457723,1675447518,rapid,600+2,chess,True,GHANDEEVAM2003,2729,win,NaN,platy3,2571,resigned,NaN,https://www.chess.com/openings/Queens-Pawn-Ope...
3,https://www.chess.com/game/live/69187254489,1675449636,rapid,600+2,chess,True,GGuseinov,2674,win,NaN,GHANDEEVAM2003,2708,resigned,NaN,https://www.chess.com/openings/Sicilian-Defens...
4,https://www.chess.com/game/live/69189396913,1675451348,rapid,600+2,chess,True,GHANDEEVAM2003,2720,win,NaN,BillieKimbah,2590,resigned,NaN,https://www.chess.com/openings/Queens-Gambit-D...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,https://www.chess.com/game/live/71343349385,1677604643,blitz,180+1,chess,True,GHANDEEVAM2003,3047,resigned,NaN,baki83,2936,win,NaN,https://www.chess.com/openings/English-Opening...
114,https://www.chess.com/game/live/71343944303,1677605280,blitz,180+1,chess,True,GMLazaroBruzon,2914,timeout,NaN,GHANDEEVAM2003,3052,win,NaN,https://www.chess.com/openings/Pirc-Defense......
115,https://www.chess.com/game/live/71345012513,1677606367,blitz,180+1,chess,True,champ2005,3013,win,NaN,GHANDEEVAM2003,3043,resigned,NaN,https://www.chess.com/openings/Modern-Defense-...
116,https://www.chess.com/game/live/71345602853,1677606820,blitz,180+1,chess,True,GHANDEEVAM2003,3048,win,NaN,Zhigalko_Sergei,2879,timeout,NaN,https://www.chess.com/openings/English-Opening...


In [ ]:
# Tests for Exercise 2
assert latest_game == df_old["end_time"].max(), "latest_game should be the most recent end_time in df_old"
assert type(df_all["end_time"].iloc[0]) == pd.Timestamp, 'Convert "end_time" to dates in both df_old and df_new'
assert (df_all["time_control"].apply(type) == str).all(), 'Every value of "time_control" should be text: convert df_old["time_control"] with .astype(str)'
assert set(df_all.columns) == set(df_old.columns), "df_all should have the same columns as df_old"
assert df_all["url"].is_unique, 'df_all has duplicate games: use .drop_duplicates(subset = "url")'
assert df_old["url"].isin(df_all["url"]).all(), "Every game in df_old should also be in df_all"
assert len(df_all) - len(df_old) == (~df_new["url"].isin(df_old["url"])).sum(), \
    "The number of new games should equal the number of games in df_new that are not in df_old"

print("All tests for Exercise 2 passed!")

<font size = "4">

**Exercise 3 (recoding results):** The "white_result" and "black_result" columns have many different values: "win", "resigned", "timeout", "checkmated", "agreed", "repetition", ...

- Write a function `group_result(result)` that returns:
    - "win" if `result` is "win"
    - "draw" if `result` is one of "agreed", "repetition", "stalemate", "insufficient", "50move", "timevsinsufficient"
    - "loss" for anything else
- Create `df_white`, the games in `df_all` where your player had the white pieces, with `df_all[df_all["white_username"] == username].copy()`. Create `df_black` the same way. (`.copy()` makes a separate DataFrame, so adding a column to it doesn't affect `df_all`.)
- Add a column "my_result" to each: use `.apply(group_result)` on "white_result" for `df_white` and on "black_result" for `df_black`.
- For each DataFrame, print the percentage of wins, draws, and losses: `(100 * df_white["my_result"].value_counts(normalize = True)).round(1)`.

In [ ]:
# your code here

In [ ]:
# Tests for Exercise 3
assert group_result("win") == "win", 'group_result("win") should be "win"'
for result in ["agreed", "repetition", "stalemate", "insufficient", "50move", "timevsinsufficient"]:
    assert group_result(result) == "draw", f'group_result("{result}") should be "draw"'
for result in ["resigned", "timeout", "checkmated", "abandoned"]:
    assert group_result(result) == "loss", f'group_result("{result}") should be "loss"'

assert (df_white["white_username"] == username).all(), "df_white should only contain games where your player had the white pieces"
assert (df_black["black_username"] == username).all(), "df_black should only contain games where your player had the black pieces"
assert len(df_white) + len(df_black) == len(df_all), \
    "Every game should be in df_white or df_black. Is username spelled exactly as in the data?"
assert set(df_white["my_result"]) <= {"win", "draw", "loss"}, 'df_white["my_result"] should only contain "win", "draw", and "loss"'
assert set(df_black["my_result"]) <= {"win", "draw", "loss"}, 'df_black["my_result"] should only contain "win", "draw", and "loss"'

print("All tests for Exercise 3 passed!")

<font size = "4">

**Exercise 4 (stats for many GMs):** "chess_raw_data/titled_GM.json" was downloaded from the `/titled/GM` path and lists every Grandmaster (GM) account on chess.com.

- Load the file with `open` and `json.load` into a variable called `titled_gm`.
- Store `titled_gm["players"]` in a variable called `gm_usernames`, and print its length.

In [ ]:
# your code here

<font size = "4">

Requesting stats for every GM would take about half an hour, so we'll use a random sample of 100 GMs instead. The cell below creates it for you, as a list called `gm_sample`.

`random.seed(185)` makes Python pick the same "random" sample every time the cell runs, so everyone in the class gets the same 100 GMs.

In [ ]:
# You don't need to write anything in this cell: just run it. 
import random

random.seed(185)    # the same "random" sample every time this cell runs
num_gms = 100
gm_sample = random.sample(gm_usernames, num_gms)

print(gm_sample[:5])

<font size = "4">

Now request the stats of every GM in the sample and summarize them:

- Create an empty list `all_rows` and a counter `num_skipped = 0`. Loop over `gm_sample` (call the loop variable `gm`, so you don't overwrite `username`). For each GM:
    - Request their stats with `my_client.get_stats(gm)`.
    - If the result is `None` (the request failed), add 1 to `num_skipped`.
    - Otherwise, add `stats_to_rows(gm, stats)` to `all_rows`. Some accounts are missing parts of their stats, which raises a `KeyError`: use `try`/`except KeyError` to add 1 to `num_skipped` instead.
- Create a DataFrame called `df_gm_ratings` from `all_rows`. Print the number of GMs in it, `df_gm_ratings["username"].nunique()`, and print `num_skipped`.
- Group `df_gm_ratings` by "time_class" and use `.agg` to compute the number of GMs (`("username", len)`) and the mean rating (`("rating", "mean")`) for each time class. Name the columns "players" and "mean_rating", round to 1 decimal, store the result in `gm_agg`, and display it.
- Store the 5 rows of `df_gm_ratings` with the highest blitz rating in `top_blitz`, and display it.

The loop makes 100 requests, so it takes a minute or two.

In [ ]:
# your code here


In [ ]:
# Tests for Exercise 4
assert gm_usernames == titled_gm["players"], 'gm_usernames should be titled_gm["players"]'
assert len(gm_sample) == num_gms and set(gm_sample) <= set(gm_usernames), \
    "Run the cell that creates gm_sample after creating gm_usernames"
assert set(df_gm_ratings["username"]) <= set(gm_sample), "df_gm_ratings should only contain GMs from gm_sample"
assert df_gm_ratings["username"].nunique() + num_skipped <= num_gms, \
    "A GM can't be both in df_gm_ratings and skipped: check where you add 1 to num_skipped"
assert num_skipped < num_gms, "Every GM was skipped: is the internet working?"
assert list(df_gm_ratings.columns) == ["username", "time_class", "rating", "rd", "rating_date",
                                       "best_rating", "wins", "losses", "draws"], \
    "df_gm_ratings should have the columns made by stats_to_rows"

assert list(gm_agg.columns) == ["players", "mean_rating"], 'gm_agg should have the columns "players" and "mean_rating"'
for time_class in gm_agg.index:
    in_class = df_gm_ratings[df_gm_ratings["time_class"] == time_class]
    assert gm_agg.loc[time_class, "players"] == len(in_class), f'"players" is wrong for {time_class}'
    assert gm_agg.loc[time_class, "mean_rating"] == round(in_class["rating"].mean(), 1), f'"mean_rating" is wrong for {time_class}'

assert len(top_blitz) == min(5, (df_gm_ratings["time_class"] == "blitz").sum()), "top_blitz should have 5 rows"
assert (top_blitz["time_class"] == "blitz").all(), "top_blitz should only contain blitz ratings"
assert list(top_blitz["rating"]) == sorted(df_gm_ratings.query("time_class == 'blitz'")["rating"], reverse = True)[:5], \
    "top_blitz should contain the 5 highest blitz ratings, from highest to lowest"

print("All tests for Exercise 4 passed!")

<font size = "4">

**Exercise 5 (streaming platforms):** A player's profile can include the key "streaming_platforms": a list with one dictionary per platform they stream on. For example, Hikaru's is

`[{"type": "twitch", "channel_url": "https://twitch.tv/gmhikaru"}]`

Players who don't stream have an empty list, or no "streaming_platforms" key at all.

**Part A:** The cell below creates a class `StreamingClient` that **inherits** from your `MyChessComClient`, so it already has `get_games` and everything from `ChessComClient`. It adds one method, `get_streaming_platforms(username)`, which should return a list of the platform types, e.g. `["twitch"]` for Hikaru.

- Fill in the missing loop: go through `profile.get("streaming_platforms", [])` and append the "type" of each dictionary to `platforms`.
- Using `.get` with the default `[]` means a profile with no "streaming_platforms" key gives an empty list instead of a `KeyError`.

The rest of the cell creates `my_client` from the new class and prints the platforms of Hikaru and Arjun.

In [ ]:
class StreamingClient(MyChessComClient):
    """MyChessComClient with an extra method for streaming platforms."""

    def get_streaming_platforms(self, username):
        """List of the platforms (e.g. "twitch") a player streams on. None if the request fails."""
        profile = self.get_profile(username)
        if profile is None:
            return None

        platforms = []
        # your code here: loop over profile.get("streaming_platforms", []) and append each "type" to platforms

        return platforms


my_client = StreamingClient("DATASCI185-course-notebook (Emory University)")

print(my_client.get_streaming_platforms("Hikaru"))
print(my_client.get_streaming_platforms("GHANDEEVAM2003"))

<font size = "4">

**Part B:** Collect the platforms of every GM in `gm_sample` from Exercise 4.

- Create an empty list `platform_list`.
- Loop over `gm_sample`. For each GM, call `my_client.get_streaming_platforms(gm)`. If the result is not `None`, add its entries to `platform_list` (a GM who streams on both Twitch and YouTube adds both). Hint: `list_1 + list_2` joins two lists.
- Print `len(platform_list)`.

This makes 100 requests, so it takes a minute or two. If `platform_list` is empty, none of the GMs in your sample stream: increase `num_gms` in Exercise 4 and run Exercises 4 and 5 again.

In [ ]:
# your code here

<font size = "4">

**Part C:** Rank the platforms from most to least popular.

- Create a DataFrame called `df_platforms` with one column, "platform", from `platform_list`: `pd.DataFrame({"platform": platform_list})`.
- Store `df_platforms["platform"].value_counts()` in a variable called `platform_counts`, and display it. `value_counts` sorts the platforms from most to least common.
- Print the most popular platform, `platform_counts.index[0]`, and the least popular, `platform_counts.index[-1]`.

In [ ]:
# your code here


<font size = "4">

**Part D:** Make a horizontal bar chart of the ranking.

- Use `plt.barh(platform_counts.index, platform_counts.values)`.
- Label the x-axis "Number of GMs" and add the title "Streaming platforms of chess.com GMs".

In [ ]:
# your code here


<font size = "4">

**Exercise 6 (Formula One API):** The Jolpica API returns at most 100 rows per request. Use `requests.get` with `headers = HEADERS` to request the first page of 2025 race results, and call the result `response`:

```
https://api.jolpi.ca/ergast/f1/2025/results.json?limit=100&offset=0
```

- Print the status code.
- Store `response.json()["MRData"]` in a variable called `mrdata`, and print its "limit", "offset", and "total".
- Store `mrdata["RaceTable"]["Races"]` in a variable called `races`, and print the number of races in it.
- Write a loop that adds up `len(race["Results"])` for every race in `races`, and print the total number of results in this page.

In [ ]:
# your code here

<font size = "4">

**Exercise 7 (Formula One API):** Request **every** page of the 2025 results, and find the constructor with the most points.

- Write a function `get_season_results(season)` that returns a list of pages (the JSON of each response). Use a `while` loop: start with `offset = 0` and `total = 1`; in each step, request `f"https://api.jolpi.ca/ergast/f1/{season}/results.json?limit=100&offset={offset}"`, append `response.json()` to the list, set `total = int(page["MRData"]["total"])`, add 100 to `offset`, and pause with `time.sleep(0.5)`. Stop when `offset` is no longer less than `total`.
- Call `pages_2025 = get_season_results(2025)` and print `len(pages_2025)`.
- Loop over every page, every race in the page, and every result in the race, to create a DataFrame called `df_2025` with the columns "round", "raceName", "driverId", "constructorId", "grid", "position", and "points". Convert "round", "grid", and "position" to `int` and "points" to `float`.
- Print `len(df_2025)`. It should equal "total" from Exercise 6.
- Group by "constructorId", add up "points", and sort in descending order. Display the result.

Note: a page can end in the middle of a race, so the same race can appear on two pages, each with some of its results. Looping over every page, race, and result handles this.

If the API is unavailable, load "f1_api_data/f1_cache.json" with `open` and `json.load`. It is a dictionary whose keys are the URLs above (for offsets 0, 100, 200, 300, 400) and whose values are the JSON you would have received.

In [ ]:
# your code here